In [1]:
import os, json, shutil
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import load_metric

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


c:\Users\Maryam\Desktop\Urdu-Multi-Domain-Script-Research\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html  from .autonotebook import tqdm as notebook_tqdm

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX A4000


In [2]:
ROOT = os.path.abspath(os.path.join('..', '..'))
BASE = os.path.join(ROOT, 'Nastaliq', 'QA')

# Domain A: Religious Text (Hadith / Islamic Traditions)
qa_a_train = pd.read_csv(os.path.join(BASE, 'Domain_A_Religious_Hadith', 'train.csv'), encoding='utf-8-sig')
qa_a_test = pd.read_csv(os.path.join(BASE, 'Domain_A_Religious_Hadith', 'test.csv'), encoding='utf-8-sig')

# Domain B: General Knowledge
qa_b_train = pd.read_csv(os.path.join(BASE, 'Domain_B_General_Knowledge', 'train.csv'), encoding='utf-8-sig')
qa_b_test = pd.read_csv(os.path.join(BASE, 'Domain_B_General_Knowledge', 'test.csv'), encoding='utf-8-sig')

# Domain C: Long-Form Multi-Domain QnA
qa_c_train = pd.read_csv(os.path.join(BASE, 'Domain_C_Long_Form_QnA', 'train.csv'), encoding='utf-8-sig')
qa_c_test = pd.read_csv(os.path.join(BASE, 'Domain_C_Long_Form_QnA', 'test.csv'), encoding='utf-8-sig')

# Domain D: Short-Form Multi-Domain QnA
qa_d_train = pd.read_csv(os.path.join(BASE, 'Domain_D_Short_Form_QnA', 'train.csv'), encoding='utf-8-sig')
qa_d_test = pd.read_csv(os.path.join(BASE, 'Domain_D_Short_Form_QnA', 'test.csv'), encoding='utf-8-sig')

print('Domain sizes (train / test):')
for name, tr, te in [
    ('A_Religious_Hadith',       qa_a_train, qa_a_test),
    ('B_General_Knowledge',              qa_b_train, qa_b_test),
    ('C_Long_Form_QnA',        qa_c_train, qa_c_test),
    ('D_Short_Form_QnA',       qa_d_train, qa_d_test),
]:
    print(f'  {name}: train={len(tr)}  test={len(te)}  pairs={len(tr)+len(te)}')


Domain sizes (train / test):
  A_Religious_Hadith: train=820  test=205  pairs=1025
  B_General_Knowledge: train=4910  test=1228  pairs=6138
  C_Long_Form_QnA: train=648  test=162  pairs=810
  D_Short_Form_QnA: train=2450  test=613  pairs=3063


In [3]:
XLM_MODEL_ID = 'xlm-roberta-base'
BERT_MODEL_ID = 'bert-base-multilingual-cased'

MAX_LEN = 384  # Increased for QA tasks
MAX_TRAIN = 5000
EPOCHS = 3
PATIENCE = 2
BATCH_TRAIN = 4
BATCH_EVAL = 8
LR = 2e-5

RESULTS_BASE  = os.path.join(ROOT, 'results', 'T1_Nastaliq_QA')
os.makedirs(RESULTS_BASE, exist_ok=True)

domains = [
    ('A_Religious_Hadith',       qa_a_train, qa_a_test),
    ('B_General_Knowledge',              qa_b_train, qa_b_test),
    ('C_Long_Form_QnA',        qa_c_train, qa_c_test),
    ('D_Short_Form_QnA',       qa_d_train, qa_d_test),
]


In [ ]:
def cap_dataset(df, max_samples=MAX_TRAIN):
    if len(df) <= max_samples:
        return df
    capped = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return capped.head(max_samples)


class UrduQADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.questions = df['question'].astype(str).tolist()
        self.answers = df['answer'].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        answer = self.answers[idx]

        encoding = self.tokenizer(
            question,
            answer,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor([0], dtype=torch.long)  # Placeholder for QA loss
        }


# For extractive QA, we need to use a different approach
# Here we'll use a simplified approach for QA experiments
def prepare_qa_examples(question, answer, max_len=MAX_LEN, tokenizer=None):
    if tokenizer is None:
        # Return a simple placeholder if no tokenizer provided
        return {
            'input_ids': torch.tensor([[0]]),  # placeholder
            'attention_mask': torch.tensor([[1]]),  # placeholder
        }
    
    # Tokenize the question-answer pair
    encoding = tokenizer(
        question,
        answer,
        truncation=True,
        padding='max_length',
        max_length=max_len,
        return_tensors='pt'
    )

    return {
        'input_ids': encoding['input_ids'].squeeze(),
        'attention_mask': encoding['attention_mask'].squeeze(),
    }


# Simple metrics for QA
def compute_qa_metrics(eval_pred):
    # Simplified metrics - for real QA, use SQuAD metrics
    # Here we'll use a placeholder that just returns the predictions
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # For QA, typically we'd use Exact Match or F1 score
    # This is a placeholder - in real implementation, use SQuAD metrics
    return {
        'exact_match': float(np.mean(preds == labels)),
        'f1': float(np.mean(preds == labels))  # Placeholder
    }


def run_qa_experiment(model_id, model_label, src_name, train_df, tgt_name, test_df):
    out_dir     = os.path.join(RESULTS_BASE, model_label)
    os.makedirs(out_dir, exist_ok=True)
    result_path = os.path.join(out_dir, f'{src_name}__vs__{tgt_name}.json')

    if os.path.exists(result_path):
        print(f'  [SKIP] {src_name} -> {tgt_name} already done.')
        with open(result_path, encoding='utf-8') as f:
            return json.load(f)['exact_match'], None

    run_type = 'IN-DOMAIN' if src_name == tgt_name else 'CROSS-DOMAIN'
    capped   = cap_dataset(train_df)
    print(f'\n  [{run_type}] Train: {src_name} ({len(capped)}) -> Test: {tgt_name} ({len(test_df)})')

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModelForQuestionAnswering.from_pretrained(model_id)

    # For QA, we'll create a simple training loop
    # In practice, QA requires extractive reading comprehension
    # Here we'll simulate the training with a simplified approach

    # Create dataset with question-answer pairs
    train_dataset = UrduQADataset(capped, tokenizer)
    val_dataset = UrduQADataset(capped.sample(max(int(len(capped) * 0.1), 1), random_state=42), tokenizer)

    out_dir = os.path.join(out_dir, f'_ckpt_{src_name}_{tgt_name}')

    args = TrainingArguments(
        output_dir                  = out_dir,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_TRAIN,
        per_device_eval_batch_size  = BATCH_EVAL,
        learning_rate               = LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'exact_match',
        greater_is_better           = True,
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
        report_to                   = 'none',
        save_total_limit            = 1,
    )

    # For QA, we'll use a simplified trainer approach
    # Note: Real QA requires extractive reading comprehension setup
    print(f'  Note: QA experiment requires extractive QA setup. This is a simplified placeholder.')

    # Create a simple result for demonstration
    exact_match = 0.0  # Placeholder
    f1_score = 0.0    # Placeholder

    result = {
        'task': 'T1_Nastaliq_QA', 'model': model_label, 'model_id': model_id,
        'source': src_name, 'target': tgt_name, 'type': run_type,
        'train_size': len(capped), 'test_size': len(test_df),
        'exact_match': round(exact_match, 4), 'f1': round(f1_score, 4),
        'note': 'Simplified QA placeholder - real implementation requires extractive QA setup',
        'classification_report': {'exact_match': round(exact_match, 4), 'f1': round(f1_score, 4)}
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)

    print(f'  exact-match={exact_match:.4f}  f1={f1_score:.4f}')
    return exact_match, None